In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import json
import os

from transformers import BlipProcessor, BlipForConditionalGeneration
from torch.optim import AdamW
from tqdm import tqdm

In [ ]:
device="cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:

#1 Load the processor from the official SalesForce repo
processor=BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")

#2 Load the actual model weights from the custom repo
model= BlipForConditionalGeneration.from_pretrained("utkarshpise/blip-fedrated_1")

model.to(device)

In [ ]:
from huggingface_hub import login
login()  # paste your HF token

In [ ]:
from google.colab import files
files.upload()

In [ ]:
!kaggle datasets download -d thedevastator/rsicd-image-caption-dataset
!unzip *.zip

In [ ]:
import pandas as pd
from torch.utils.data import Dataset
from PIL import Image
from io import BytesIO
import ast  # ✅ safer than eval
# remove json import (not needed)

class RSICD_CSV_Dataset(Dataset):
    def __init__(self, csv_file, processor):
        self.df = pd.read_csv(csv_file)
        self.processor = processor

        self.samples = []

        for _, row in self.df.iterrows():
            captions = ast.literal_eval(row["captions"])  # ✅ safe

            image_dict = ast.literal_eval(row["image"])   # ✅ fix here
            image_bytes = image_dict["bytes"]

            for caption in captions:
                self.samples.append((image_bytes, caption))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        image_bytes, caption = self.samples[idx]

        image = Image.open(BytesIO(image_bytes)).convert("RGB")
        caption = caption.lower().strip()

        encoding = self.processor(
            images=image,
            text=caption,
            padding="max_length",
            truncation=True,
            max_length=30,
            return_tensors="pt"
        )

        encoding = {k: v.squeeze(0) for k, v in encoding.items()}
        encoding["labels"] = encoding["input_ids"]

        return encoding

In [ ]:
from torch.utils.data import DataLoader

train_dataset = RSICD_CSV_Dataset("train.csv", processor)
val_dataset   = RSICD_CSV_Dataset("valid.csv", processor)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=8)

In [ ]:
from torch.optim import AdamW
from tqdm import tqdm

optimizer = AdamW(model.parameters(), lr=3e-5)
EPOCHS = 3

for epoch in range(EPOCHS):

    # 🔹 TRAINING
    model.train()
    train_loss = 0

    for batch in tqdm(train_loader, desc=f"Train Epoch {epoch+1}"):
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model(**batch)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)

    # 🔹 VALIDATION
    model.eval()
    val_loss = 0

    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Val Epoch {epoch+1}"):
            batch = {k: v.to(device) for k, v in batch.items()}

            outputs = model(**batch)
            val_loss += outputs.loss.item()

    avg_val_loss = val_loss / len(val_loader)

    print(f"""
    Epoch {epoch+1}
    Train Loss: {avg_train_loss:.4f}
    Val Loss:   {avg_val_loss:.4f}
    """)

In [ ]:
import ast

def generate_caption(row):
    image_dict = ast.literal_eval(row["image"])   # ✅ fix
    image_bytes = image_dict["bytes"]

    image = Image.open(BytesIO(image_bytes)).convert("RGB")

    inputs = processor(images=image, return_tensors="pt").to(device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_length=30,
            num_beams=5
        )

    return processor.decode(output[0], skip_special_tokens=True)
df = pd.read_csv("test.csv")
print(generate_caption(df.iloc[0]))

In [ ]:
SAVE_PATH = "./blip_fed_round2"

model.save_pretrained(SAVE_PATH)
processor.save_pretrained(SAVE_PATH)

In [ ]:
from huggingface_hub import login

login("TOKEN")

from huggingface_hub import create_repo, upload_folder

repo_name = "OmPatil3690/blip-rsicd_1-captioning-after-f1"

create_repo(repo_name, exist_ok=True)

upload_folder(
    folder_path=SAVE_PATH,
    repo_id=repo_name,
    repo_type="model"
)

print("Uploaded via folder!")